In [16]:
# Import the C5.0 library
import sys
sys.path.insert(0, '.')  # Add current directory to path
from C5_0 import C50  # Import C50 class from C5.0.py

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Load the golf dataset
df = pd.read_csv(r"../lab_part_3/lab1/lab1/dataset/golf.txt")
print("Dataset shape:", df.shape)
print(df.head())

# Prepare data - separate features from target
X = df[['Outlook', 'Temp.', 'Humidity', 'Wind']].values
y = df['Decision'].values

# Convert categorical features to numeric using LabelEncoder
encoders = {}
X_encoded = X.copy().astype(str)

for i, col in enumerate(['Outlook', 'Temp.', 'Humidity', 'Wind']):
    le = LabelEncoder()
    X_encoded[:, i] = le.fit_transform(X[:, i].astype(str))
    encoders[col] = le

# Convert to numeric
X_encoded = X_encoded.astype(float)

# Encode target variable
y_encoder = LabelEncoder()
y_encoded = y_encoder.fit_transform(y)

# Split into train and test
X_train, y_train = X_encoded[:8], y_encoded[:8]
X_test, y_test = X_encoded[8:], y_encoded[8:]

print("\nTraining samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])
print("Features encoded successfully!")

# Create and train C5.0 model
print("\n" + "="*50)
print("C5.0 Model Training")
print("="*50)

model = C50(max_depth=5, 
           min_samples_split=2, 
           min_samples_leaf=1,
           confidence_factor=0.25, 
           criterion='entropy')

# Fit the model
model.fit(X_train, y_train, 
         feature_names=['Outlook', 'Temperature', 'Humidity', 'Wind'])

print("\nModel trained successfully!")

# Get predictions
y_pred = model.predict(X_test)

# Evaluate accuracy
accuracy = model.score(X_test, y_test)
print("\nAccuracy on TEST dataset: {:.2f}%".format(accuracy * 100))

# Print comparison
print("\nCompare predictions vs actual:")
print("-" * 50)
correct = 0
for i, (pred, actual) in enumerate(zip(y_pred, y_test)):
    pred_label = y_encoder.inverse_transform([pred])[0]
    actual_label = y_encoder.inverse_transform([actual])[0]
    match = "✓" if pred == actual else "✗"
    if pred == actual:
        correct += 1
    print("{} Sample {}: Predicted={}, Actual={}".format(match, i+1, pred_label, actual_label))

print("\nCorrect predictions: {}/{}".format(correct, len(y_test)))

# Print tree information
print("\n" + "="*50)
print("Tree Information")
print("="*50)
tree_info = model.get_tree_info()
print("Tree Depth: {}".format(tree_info['depth']))
print("Number of Nodes: {}".format(tree_info['n_nodes']))
print("Number of Leaves: {}".format(tree_info['n_leaves']))

# Print tree structure
print("\nTree Structure:")
print("-" * 50)
model.print_tree()

Dataset shape: (14, 5)
    Outlook Temp. Humidity    Wind Decision
0     Sunny   Hot     High    Weak       No
1     Sunny   Hot     High  Strong       No
2  Overcast   Hot     High    Weak      Yes
3      Rain  Mild     High    Weak      Yes
4      Rain  Cool   Normal    Weak      Yes

Training samples: 8
Testing samples: 6
Features encoded successfully!

C5.0 Model Training

Model trained successfully!

Accuracy on TEST dataset: 66.67%

Compare predictions vs actual:
--------------------------------------------------
✗ Sample 1: Predicted=No, Actual=Yes
✓ Sample 2: Predicted=Yes, Actual=Yes
✗ Sample 3: Predicted=No, Actual=Yes
✓ Sample 4: Predicted=Yes, Actual=Yes
✓ Sample 5: Predicted=Yes, Actual=Yes
✓ Sample 6: Predicted=No, Actual=No

Correct predictions: 4/6

Tree Information
Tree Depth: 3
Number of Nodes: 7
Number of Leaves: 4

Tree Structure:
--------------------------------------------------
If Feature_0 <= 1.00:
  If Feature_3 <= 0.00:
    If Feature_0 <= 0.00:
      Predict: